<a href="https://colab.research.google.com/github/Divyashree-02112006/divya-codeboosters-2026/blob/main/Day-4/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark --quiet
print('Pyspark installation complete!')


Pyspark installation complete!


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


#SparkSession
spark=SparkSession.builder \
.appName('Day4_BigData_Sales')\
.config('spark.sql.adaptive.enabled','true')\
.getOrCreate()


print(f'Spark Session : {spark.version}')


Spark Session : 4.0.2


In [ ]:
df_bronze = spark.read \
.option('header','true') \
.option('inferSchema','true') \
.csv('large_sales_data.csv')

print('BRONZE LAYER-Raw Data')
print(f'Rows : {df_bronze.count()}')
print(f'Columns : {len(df_bronze.columns)}')
print(f'Names : {df_bronze.columns}')

print()
df_bronze.printSchema()

BRONZE LAYER-Raw Data
Rows : 5000
Columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print('First 5 rows:')
df_bronze.show(5,truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

print('Last 5 rows:')
df_bronze.tail(5)

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

[Row(order_id=5996, customer_name='Ananya Das', product='Mouse', category='Accessories', quantity=13, unit_price=800, revenue=10400, order_date=datetime.date(2023, 11, 18), city='Bangalore', region='South', sales_rep='Meera Patel', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id=5997, customer_name='Suresh Rao', product='Webcam', category='Accessories', quantity=9, unit_price=2500, revenue=22500, order_date=datetime.date(2023, 6, 7), city='Chennai', region='South', sales_rep='Sunita Rao', payment_method='Credit Card', order_status='Delivered'),
 Row(order_id=5998, customer_name='Arjun Nair', product='Webcam', category='Accessories', quantity=1, unit_price=2500, revenue=2500, order_date=datetime.date(2023, 4, 7), city='Jaipur', region='North', sales_rep='Kavya Reddy', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id=5999, customer_name='Arjun Nair', product='Laptop', category='Electronics', quantity=14, unit_price=45000, revenue=630000, order

In [ ]:
df_bronze.write \
.mode('overwrite') \
.parquet('sales_bronze.parquet')

print('Bronze parquet saved:sales_bronze.parquet')

Bronze parquet saved:sales_bronze.parquet


In [ ]:
import os

def get_dir_size(path):
  """get total size of a file or directory in KB."""
  if os.path.isfile(path):
    return os.path.getsize(path) / 1024
  total=0
  for dirpath,dirnames,filenames in os.walk(path):
    for f in filenames:
      total+=os.path.getsize(os.path.join(dirpath,f))
  return total/1024

csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction=(1 - parquet_size/csv_size)*100

print(f'Parquet size:{parquet_size:.1f} KB')
print(f'CSV size:{csv_size:.1f} KB')
print(f'Reduction in size:{reduction:.1f}% smaller')

Parquet size:55.1 KB
CSV size:529.3 KB
Reduction in size:89.6% smaller


In [ ]:
df_silver=df_bronze \
 .dropDuplicates() \
 .dropna(subset=['order_id','product','revenue'])

df_silver=df_silver.withColumn(
     'order_date',
     to_date(col('order_date'),'dd-MM-yyyy')
)

df_silver=df_silver \
  .withColumn('order_year',year(col('order_date'))) \
  .withColumn('order_month',month(col('order_date')))

df_silver=df_silver.withColumn(
      'revenue category',
      F.when(col('revenue')>30000,'high')
      .when(col('revenue')>10000,'medium')
      .otherwise('low')
  )
print(f'Rows : {df_silver.count()}')
print('new columns added:order_year,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue category').show(8)

Rows : 5000
new columns added:order_year,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue category|
+----------+-------+----------+-----------+----------------+
|  Keyboard|  13200|      2023|          2|          medium|
|    Webcam|  17500|      2023|          1|          medium|
|   Speaker|  58500|      2023|          4|            high|
|  Keyboard|   9600|      2023|         12|             low|
|    Laptop| 180000|      2023|          8|            high|
|Headphones|  38500|      2023|          5|            high|
|    Webcam|  35000|      2023|         11|            high|
|    Laptop| 360000|      2023|          1|            high|
+----------+-------+----------+-----------+----------------+
only showing top 8 rows


In [ ]:
df_silver.write \
.mode('overwrite') \
.parquet('sales_silver.parquet')

print('Silver parquet saved:sales_silver.parquet')
print(f'Silver size:{get_dir_size("sales_silver.parquet"):.1f} KB')

print('Bronze parquet saved:sales_bronze.parquet')
print(f'Bronze size:{get_dir_size("sales_bronze.parquet"):.1f} KB')


df_verify=spark.read.parquet('sales_silver.parquet')
print(f'Read-back rows: {df_verify.count()}(should match silver count)')
df_verify.printSchema()

Silver parquet saved:sales_silver.parquet
Silver size:59.8 KB
Bronze parquet saved:sales_bronze.parquet
Bronze size:55.1 KB
Read-back rows: 5000(should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue category: string (nullable = true)

